### **MODELING**

In this section, we will focus on building and evaluating predictive models for Remaining Useful Life (RUL):

1. **Baseline Model:**

   * We will first implement a **Linear Regression model** as a baseline.
   * This will help us evaluate the impact of scaling and feature engineering on model performance.

2. **Advanced Models:**

   * After establishing the baseline, we will model the data using more advanced algorithms:

     * **XGBoost**
     * **NGBoost**
     * **Decision Trees**
   * These models are chosen for their ability to capture non-linear relationships and complex feature interactions.

3. **Model Evaluation:**

   * The performance of each model will be assessed using regression metrics such as:

     * **Root Mean Squared Error (RMSE)**
     * **Mean Absolute Error (MAE)**
     * **R² Score**
   * This comparison will allow us to identify the best-performing model.

4. **Model Optimization:**

   * Once the best model is selected, we will perform **hyperparameter tuning** to further improve its predictive accuracy and generalization ability.

**The modeling phase will provide both a baseline and advanced approaches, enabling us to compare performance across different methods and identify the optimal model to predict RUL effectively.**




In [7]:
 # Importing necessary libraries
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge
from ngboost import NGBRegressor
from ngboost.distns import LogNormal, Poisson
from sklearn.metrics import accuracy_score, confusion_matrix , precision_score , recall_score , f1_score , r2_score
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import cross_val_score,KFold


In [8]:
 # Loading the datasets 
cleaned_test_dfs = {}
cleaned_train_dfs = {}
train_dfs = {}
test_dfs = {}

names = ['FD001','FD002','FD003','FD004']
test_cleaned_csv = ['cleaned_test FD001 with RUL.csv','cleaned_test FD002 with RUL.csv','cleaned_test FD003 with RUL.csv','cleaned_test FD004 with RUL.csv']
train_cleaned_csv = ['cleaned_train FD001 with RUL.csv','cleaned_train FD002 with RUL.csv','cleaned_train FD003 with RUL.csv','cleaned_train FD004 with RUL.csv']

test_csv = ['test FD001 with RUL.csv','test FD002 with RUL.csv','test FD003 with RUL.csv','test FD004 with RUL.csv']
train_csv = ['train FD001 with RUL.csv','train FD002 with RUL.csv','train FD003 with RUL.csv','train FD004 with RUL.csv']

 # loading the datasets
for name , df1 , df2 , df3 , df4 in zip( names, test_cleaned_csv, train_cleaned_csv, test_csv, train_csv ):
    cleaned_test_dfs[name] = pd.read_csv(df1)
    cleaned_train_dfs[name] = pd.read_csv(df2)
    test_dfs[name] = pd.read_csv(df3)
    train_dfs[name] = pd.read_csv(df4)


we will first model the unscalled and unfeatured data to see how the model performs with unscalled data then compare the model performance with the scalled data.

In [9]:
 # pipeline
# Cross-validation setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for (name1, df1) , (name2, df2)in zip(train_dfs.items(), test_dfs.items()):
    X_train = df1.iloc[ :,1:-1 ]
    y_train = df1.iloc[ :,-1 ]

    X_test = df2.iloc[ :,1:-1 ]
    y_test = df2.iloc[ :,-1 ]

     # --fitting a baseline model-- 
    reg1 = LinearRegression()
    reg1.fit(X_train, y_train )
    y_pred = reg1.predict( X_test )

    print(f'========= {name1} MODEL OUTPUTS ==========')
    print('R2 Score(Unscaled) :', r2_score( y_test, y_pred ))

     # -- Cross Validation --
    cv_scores = cross_val_score(reg1, X_train, y_train, cv=cv, scoring='r2')
    print('Linear Regression (Unscaled) Cross validation R2 Mean :', cv_scores.mean())

     # --Ridge regression for regulization--
    ridge = Ridge(alpha= 1.0 )
    ridge.fit(X_train, y_train)
    y_pred_ridge = ridge.predict(X_test)
    
    print(f'Ridge Regression R2 Score(Unscaled) :', r2_score(y_test, y_pred_ridge))
    
     # --Scalling the data--
    scaler = StandardScaler().set_output( transform = 'pandas')
    scaler.fit(X_train)

    scaled_X_train = scaler.transform(X_train)
    scaled_X_test = scaler.transform(X_test)
    
    reg2 = LinearRegression()
    reg2.fit(scaled_X_train,y_train)
    y_pred_scaled = reg2.predict(scaled_X_test)

    print('R2 Score(Scaled):', r2_score( y_test, y_pred_scaled ))
    
    # --Cross validation--
    cv_scores = cross_val_score(reg2, scaled_X_train, y_train, cv=cv, scoring='r2')
    print('Linear Regression (Scaled) Cross validation R2 Mean :', cv_scores.mean())
    
     # --Ridge regression for regulization--
    ridge = Ridge(alpha= 1.0 )
    ridge.fit(scaled_X_train, y_train)
    y_pred_ridge_scaled = ridge.predict(scaled_X_test)
    
    print(f'Ridge Regression (Scaled) R2 Score :', r2_score(y_test, y_pred_ridge))
    print('---'*50)


========= FD001 MODEL OUTPUTS ==========
R2 Score(Unscaled) : 0.6972198889255656
Linear Regression (Unscaled) Cross validation R2 Mean : 0.7653736260953764
Ridge Regression R2 Score(Unscaled) : 0.6983297483170029
R2 Score(Scaled): 0.6972198889255832
Linear Regression (Scaled) Cross validation R2 Mean : 0.76537362609538
Ridge Regression (Scaled) R2 Score : 0.6983297483170029
------------------------------------------------------------------------------------------------------------------------------------------------------
========= FD002 MODEL OUTPUTS ==========
R2 Score(Unscaled) : 0.6766257384518963
Linear Regression (Unscaled) Cross validation R2 Mean : 0.7510060200153571
Ridge Regression R2 Score(Unscaled) : 0.6773250012838155
R2 Score(Scaled): 0.6766257384518336
Linear Regression (Scaled) Cross validation R2 Mean : 0.7510060200153488
Ridge Regression (Scaled) R2 Score : 0.6773250012838155
---------------------------------------------------------------------------------------------

From the initial experiments, we observe that applying **Ridge Regression** and **scaling the data** does **not significantly alter model performance** compared to the baseline linear regression. This suggests that the underlying multicollinearity and skewness issues are not sufficiently mitigated by these techniques alone.

This finding highlights two key insights:

1. **Regularization alone is insufficient** – While Ridge Regression penalizes large coefficients to reduce the effect of multicollinearity, the predictive performance remains largely unchanged, indicating that redundant or noisy features may still be influencing the model.
2. **Scaling does not improve baseline behavior** – Although scaling standardizes feature ranges, it does not fundamentally impact the linear model’s ability to capture nonlinear degradation patterns in the data.

Given these outcomes, we will proceed by utilizing the **feature-reduced dataframes**, where low-variance columns identified during EDA have been removed. This step will help assess whether eliminating uninformative features improves signal-to-noise ratio and enhances model generalization.

The subsequent phase will involve comparing performance across reduced-feature datasets, followed by advancing to **non-linear models** (XGBoost, NGBoost, Decision Trees) that can better capture the complex degradation dynamics.


In [11]:
 # pipeline
# Cross-validation setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for (name1, df1) , (name2, df2)in zip(cleaned_train_dfs.items(), cleaned_test_dfs.items()):
    X_train = df1.iloc[ :,1:-1 ]
    y_train = df1.iloc[ :,-1 ]

    X_test = df2.iloc[ :,1:-1 ]
    y_test = df2.iloc[ :,-1 ]

     # --fitting a baseline model-- 
    reg3 = LinearRegression()
    reg3.fit(X_train, y_train )
    y_pred = reg3.predict( X_test )

    print(f'========= {name1} MODEL OUTPUTS ==========')
    print('R2 Score(Unscaled) :', r2_score( y_test, y_pred ))

     # -- Cross Validation --
    cv_scores = cross_val_score(reg3, X_train, y_train, cv=cv, scoring='r2')
    print('Linear Regression (Unscaled) Cross validation R2 Mean :', cv_scores.mean())

     # --Ridge regression for regulization--
    ridge = Ridge(alpha= 1.0 )
    ridge.fit(X_train, y_train)
    y_pred_ridge = ridge.predict(X_test)
    
    print(f'Ridge Regression R2 Score(Unscaled) :', r2_score(y_test, y_pred_ridge))
    
     # --Scalling the data--
    scaler = StandardScaler().set_output( transform = 'pandas')
    scaler.fit(X_train)

    scaled_X_train = scaler.transform(X_train)
    scaled_X_test = scaler.transform(X_test)
    
    reg4 = LinearRegression()
    reg4.fit(scaled_X_train,y_train)
    y_pred_scaled = reg4.predict(scaled_X_test)

    print('R2 Score(Scaled):', r2_score( y_test, y_pred_scaled ))
    
    # --Cross validation--
    cv_scores = cross_val_score(reg4, scaled_X_train, y_train, cv=cv, scoring='r2')
    print('Linear Regression (Scaled) Cross validation R2 Mean :', cv_scores.mean())
    
     # --Ridge regression for regulization--
    ridge = Ridge(alpha= 1.0 )
    ridge.fit(scaled_X_train, y_train)
    y_pred_ridge_scaled = ridge.predict(scaled_X_test)
    
    print(f'Ridge Regression (Scaled) R2 Score :', r2_score(y_test, y_pred_ridge))
    print('---'*50)


========= FD001 MODEL OUTPUTS ==========
R2 Score(Unscaled) : 0.6970361032923211
Linear Regression (Unscaled) Cross validation R2 Mean : 0.7631424734552641
Ridge Regression R2 Score(Unscaled) : 0.6969431081554582
R2 Score(Scaled): 0.6970361032923144
Linear Regression (Scaled) Cross validation R2 Mean : 0.763142473455264
Ridge Regression (Scaled) R2 Score : 0.6969431081554582
------------------------------------------------------------------------------------------------------------------------------------------------------
========= FD002 MODEL OUTPUTS ==========
R2 Score(Unscaled) : 0.6777829618269525
Linear Regression (Unscaled) Cross validation R2 Mean : 0.7506118061709335
Ridge Regression R2 Score(Unscaled) : 0.6774339114568133
R2 Score(Scaled): 0.6777829618270872
Linear Regression (Scaled) Cross validation R2 Mean : 0.7506118061709328
Ridge Regression (Scaled) R2 Score : 0.6774339114568133
--------------------------------------------------------------------------------------------

After re-modelling using datasets where **low-variance columns were removed**, the model performance remained largely unchanged.

This indicates that:

1. **Low-variance features contributed little to no noise** – Their removal did not enhance predictive power, suggesting they were not influential in the model’s variance.
2. **Scaling did not impact linear predictive capacity** – The persistence of similar results implies that scaling alone cannot address the inherent nonlinearities and complex degradation dynamics present in the sensor data.

Hence, both **feature reduction** and **scaling transformations** provide limited benefit when combined with linear and ridge regression models. This outcome strengthens the motivation to transition toward **nonlinear, tree-based, and ensemble approaches** (e.g., XGBoost, NGBoost, Decision Trees), which are better suited to capture complex relationships among sensor readings and engine degradation patterns.